# 04 — Recurrences and schedules

Every dated rule this library produces is **two independent decisions**:

1. **how to cut** the window into periods — the `every` argument;
2. **what to take** from each period — the `on` argument.

`bcal.schedule(...)` is those two decisions and nothing else. The eight named functions
(`last_weekday`, `month_ends`, `imm_dates`…) are one-line spellings on top, and they stay
because they say what they do to somebody who has never seen the grammar.

In [1]:
from datetime import date

import pandas as pd

import better_calendar as bcal
from better_calendar import FRI, MON, SAT, SUN, THU, TUE, WED, Nth, Weekday, periods, schedule


def show(index):
    return list(index.strftime("%Y-%m-%d"))

## 1. The engine

In [2]:
pd.DataFrame(
    [
        {"call": 'schedule(a, b, "M", "last FRI")',
         "result": show(schedule("2026-01-01", "2026-03-31", "M", "last FRI"))},
        {"call": 'schedule(a, b, "Q", "2 THU")',
         "result": show(schedule("2026-01-01", "2026-12-31", "Q", "2 THU"))},
        {"call": 'schedule(a, b, "M", "last B", cal="XNYS")',
         "result": show(schedule("2026-01-01", "2026-03-31", "M", "last B", cal="XNYS"))},
        {"call": 'schedule(a, b, "M", "3 WED", months=(3,6,9,12))',
         "result": show(
             schedule("2026-01-01", "2026-12-31", "M", "3 WED", months=(3, 6, 9, 12))
         )},
    ]
).set_index("call")

,result
call,
"schedule(a, b, ""M"", ""last FRI"")","[2026-01-30, 2026-02-27, 2026-03-27]"
"schedule(a, b, ""Q"", ""2 THU"")","[2026-01-08, 2026-04-09, 2026-07-09, 2026-10-08]"
"schedule(a, b, ""M"", ""last B"", cal=""XNYS"")","[2026-01-30, 2026-02-27, 2026-03-31]"
"schedule(a, b, ""M"", ""3 WED"", months=(3,6,9,12))","[2026-03-18, 2026-06-17, 2026-09-16, 2026-12-16]"


### The `every` grammar — how to cut

A **bare unit** aligns to the calendar; a **multiple** anchors on `start`. That is the whole
difference between `"Q"` and `"3M"`, which are both three months long.

In [3]:
# Same window, same period length, two different cuts.
print('"Q"  calendar quarter          :',
      show(schedule("2026-02-01", "2026-12-31", "Q", "1")))
print('"3M" anchored on 1 February    :',
      show(schedule("2026-02-01", "2026-12-31", "3M", "1")))

"Q"  calendar quarter          : ['2026-04-01', '2026-07-01', '2026-10-01']
"3M" anchored on 1 February    : ['2026-02-01', '2026-05-01', '2026-08-01', '2026-11-01']


So there is **no** `anchor` parameter: the choice is already carried by the frequency
string. `"D"`, `"W"`, `"M"`, `"Q"`, `"Y"` and their multiples cover everything.

### The `on` grammar — what to take

In [4]:
window = ("2026-01-01", "2026-03-31")
pd.DataFrame(
    [
        {"on": o, "selection": show(schedule(*window, "M", o))}
        for o in ("1", "15", "last", "-2", "1 B", "last B",
                  "1st FRI", "2 THU", "last FRI", "-2 WED")
    ]
).set_index("on")

,selection
on,
1,"[2026-01-01, 2026-02-01, 2026-03-01]"
15,"[2026-01-15, 2026-02-15, 2026-03-15]"
last,"[2026-01-31, 2026-02-28, 2026-03-31]"
-2,"[2026-01-30, 2026-02-27, 2026-03-30]"
1 B,"[2026-01-01, 2026-02-02, 2026-03-02]"
last B,"[2026-01-30, 2026-02-27, 2026-03-31]"
1st FRI,"[2026-01-02, 2026-02-06, 2026-03-06]"
2 THU,"[2026-01-08, 2026-02-12, 2026-03-12]"
last FRI,"[2026-01-30, 2026-02-27, 2026-03-27]"


Negative counts from the end throughout. Ordinal suffixes are cosmetic: `"2 THU"` and
`"2nd THU"` read the same. For code rather than configuration, the typed form survives a
rename:

In [5]:
print(show(schedule(*window, "M", Nth(-1, FRI))) == show(schedule(*window, "M", "last FRI")))
print(show(schedule(*window, "M", Nth(1, "B"))) == show(schedule(*window, "M", "1 B")))
print("Nth(-1, FRI) spells back to :", str(Nth(-1, FRI)))

# parse_selector exposes the parsed form, useful for validating a config before running it.
print("parse_selector('2nd THU') :", bcal.parse_selector("2nd THU"))
print("parse_selector('edges')   :", bcal.parse_selector("edges"), "— the bcal.EDGES sentinel")
print("memoised                  :", bcal.parse_selector("last") is bcal.parse_selector("last"))

True
True
Nth(-1, FRI) spells back to : last FRI
parse_selector('2nd THU') : 2 THU
parse_selector('edges')   : Edges() — the bcal.EDGES sentinel
memoised                  : True


### Business days **or** calendar days: two independent axes

This is where the mistakes live. `on="last B"` **counts** business days; `roll=` **moves** a
result onto one. They agree more often than not, which is exactly why the difference has to
be written down rather than inferred.

In [6]:
pd.DataFrame(
    [
        {"spelling": 'on="last B"', "meaning": "last BUSINESS day of the month",
         "January 2026": show(
             schedule("2026-01-01", "2026-01-31", "M", "last B", cal="XNYS"))},
        {"spelling": 'on="last", roll="P"', "meaning": "last calendar day, PULLED BACK",
         "January 2026": show(
             schedule("2026-01-01", "2026-01-31", "M", "last", cal="XNYS", roll="P"))},
        {"spelling": 'on="3 B"', "meaning": "3rd BUSINESS day of the month",
         "January 2026": show(
             schedule("2026-01-01", "2026-01-31", "M", "3 B", cal="XNYS"))},
        {"spelling": 'on="3", roll="F"', "meaning": "3 January, pushed forward",
         "January 2026": show(
             schedule("2026-01-01", "2026-01-31", "M", "3", cal="XNYS", roll="F"))},
    ]
).set_index("spelling")

,meaning,January 2026
spelling,,
"on=""last B""",last BUSINESS day of the month,[2026-01-30]
"on=""last"", roll=""P""","last calendar day, PULLED BACK",[2026-01-30]
"on=""3 B""",3rd BUSINESS day of the month,[2026-01-06]
"on=""3"", roll=""F""","3 January, pushed forward",[2026-01-05]


The first two rows agree; the last two do not. As soon as the ordinal is no longer the last
one, the two spellings part company — which is why you have to say which you meant.

### Missing occurrences: `missing=`

In [7]:
pd.DataFrame(
    [
        {"missing": "skip (default)",
         "months kept": len(schedule("2026-01-01", "2026-12-31", "M", "31")),
         "January → April": show(schedule("2026-01-01", "2026-04-30", "M", "31"))},
        {"missing": "clamp",
         "months kept": len(
             schedule("2026-01-01", "2026-12-31", "M", "31", missing="clamp")),
         "January → April": show(
             schedule("2026-01-01", "2026-04-30", "M", "31", missing="clamp"))},
    ]
).set_index("missing")

,months kept,January → April
missing,,
skip (default),7,"[2026-01-31, 2026-03-31]"
clamp,12,"[2026-01-31, 2026-02-28, 2026-03-31, 2026-04-30]"


`clamp` makes "payable on the 31st of each month" expressible in a single call. `raise`
refuses, naming the offending period:

In [8]:
try:
    schedule("2026-01-01", "2026-12-31", "M", "31", missing="raise")
except bcal.ScheduleError as exc:
    print(exc)

The period starting 2026-02-01 has no day 31, and missing='raise' forbids skipping it. Use missing='skip' to drop such periods, or missing='clamp' to take the nearest occurrence that does exist.


### Several selectors at once

In [9]:
print("the 1st and the 15th :", show(schedule("2026-01-01", "2026-02-28", "M", ["1", "15"])))
print("first and last business day :",
      show(schedule("2026-01-01", "2026-02-28", "M", ["1 B", "last B"], cal="XNYS")))

the 1st and the 15th : ['2026-01-01', '2026-01-15', '2026-02-01', '2026-02-15']
first and last business day : ['2026-01-02', '2026-01-30', '2026-02-02', '2026-02-27']


## 2. The two founding examples

The questions that motivated the library, in their named spelling:

In [10]:
last_fridays = bcal.last_weekday("2026-01-01", "2026-12-31", FRI)
pd.DataFrame(
    {"date": show(last_fridays), "day": last_fridays.strftime("%A")}
).set_index("date")

,day
date,
2026-01-30,Friday
2026-02-27,Friday
2026-03-27,Friday
2026-04-24,Friday
2026-05-29,Friday
2026-06-26,Friday
2026-07-31,Friday
2026-08-28,Friday
2026-09-25,Friday


In [11]:
show(bcal.nth_weekday("2026-01-01", "2026-12-31", 2, THU, freq="Q"))

['2026-01-08', '2026-04-09', '2026-07-09', '2026-10-08']

Each named function **is** its generic spelling — a test pins it, otherwise the
documentation would be fiction:

In [12]:
window = ("2026-01-01", "2026-12-31")
equivalences = [
    ("last_weekday(a, b, FRI)", bcal.last_weekday(*window, FRI),
     'schedule(a, b, "M", "last FRI")', schedule(*window, "M", "last FRI")),
    ("month_ends(a, b)", bcal.month_ends(*window),
     'schedule(a, b, "M", "last")', schedule(*window, "M", "last")),
    ("month_ends(a, b, cal=…)", bcal.month_ends(*window, cal="XNYS"),
     'schedule(a, b, "M", "last B", cal=…)', schedule(*window, "M", "last B", cal="XNYS")),
    ("quarter_ends(a, b)", bcal.quarter_ends(*window),
     'schedule(a, b, "Q", "last")', schedule(*window, "Q", "last")),
    ("imm_dates(a, b)", bcal.imm_dates(*window),
     'schedule(a, b, "M", "3 WED", months=…)',
     schedule(*window, "M", "3 WED", months=(3, 6, 9, 12))),
]
pd.DataFrame(
    [{"named": n, "generic": g, "identical": show(a) == show(b)}
     for n, a, g, b in equivalences]
).set_index("named")

,generic,identical
named,,
"last_weekday(a, b, FRI)","schedule(a, b, ""M"", ""last FRI"")",True
"month_ends(a, b)","schedule(a, b, ""M"", ""last"")",True
"month_ends(a, b, cal=…)","schedule(a, b, ""M"", ""last B"", cal=…)",True
"quarter_ends(a, b)","schedule(a, b, ""Q"", ""last"")",True
"imm_dates(a, b)","schedule(a, b, ""M"", ""3 WED"", months=…)",True


## 3. Three conventions worth knowing

### `n` is 1-based, and negative counts from the end

That is the whole point: "the last Friday of the month" is what people actually ask for,
and computing it by hand is precisely where the bugs are.

In [13]:
pd.DataFrame(
    [{"n": n, "result": show(bcal.nth_weekday("2026-01-01", "2026-01-31", n, FRI))}
     for n in (1, 2, 3, 4, 5, -1, -2, -5)]
).set_index("n")

,result
n,
1,[2026-01-02]
2,[2026-01-09]
3,[2026-01-16]
4,[2026-01-23]
5,[2026-01-30]
-1,[2026-01-30]
-2,[2026-01-23]
-5,[2026-01-02]


### A missing occurrence is skipped silently

February rarely has a fifth Friday. Raising would make the API unusable over any real span
— hence `missing="skip"` by default, and `"clamp"` when you want the nearest one instead.

In [14]:
fifths = bcal.nth_weekday("2026-01-01", "2026-12-31", 5, FRI)
print(f"{len(fifths)} months out of 12 have a fifth Friday:")
print(show(fifths))
print("\nwith clamp, February falls back on the fourth:",
      show(schedule("2026-02-01", "2026-02-28", "M", "5 FRI", missing="clamp")))

4 months out of 12 have a fifth Friday:
['2026-01-30', '2026-05-29', '2026-07-31', '2026-10-30']

with clamp, February falls back on the fourth: ['2026-02-27']


### The occurrence belongs to the period, not to the window

"The last Friday of January" is a property of January. Asking from the 15th still returns
the 30th; an occurrence falling before the window is filtered out.

In [15]:
print("from 15 January, last Friday :",
      show(bcal.last_weekday("2026-01-15", "2026-01-31", FRI)))
print("from 15 January, first Friday:",
      show(bcal.nth_weekday("2026-01-15", "2026-01-31", 1, FRI)))

from 15 January, last Friday : ['2026-01-30']
from 15 January, first Friday: []


## 4. The weekday constants

An `IntEnum` aligned on `date.weekday()` — Monday is 0, so they work anywhere the standard
library expects a weekday index.

In [16]:
print("values :", {d.name: int(d) for d in Weekday})
print("date.weekday() of 31 July 2026 :", date(2026, 7, 31).weekday(), "==", int(FRI))
print("a weekend :", int(SAT), int(SUN))
print("2nd Tuesday of each quarter :",
      show(bcal.nth_weekday("2026-01-01", "2026-12-31", 2, TUE, freq="Q")))
print("last Saturdays of the quarter :",
      show(bcal.last_weekday("2026-01-01", "2026-06-30", SAT, freq="Q")))

values : {'MON': 0, 'TUE': 1, 'WED': 2, 'THU': 3, 'FRI': 4, 'SAT': 5, 'SUN': 6}
date.weekday() of 31 July 2026 : 4 == 4
a weekend : 5 6
2nd Tuesday of each quarter : ['2026-01-13', '2026-04-14', '2026-07-14', '2026-10-13']
last Saturdays of the quarter : ['2026-03-28', '2026-06-27']


## 5. The recurrences that have a name

In [17]:
print("1st day of the month     :", show(bcal.nth_day("2026-01-01", "2026-03-31", 1)))
print("last day of the month    :", show(bcal.nth_day("2026-01-01", "2026-03-31", -1)))
print("1st NYSE business day    :",
      show(bcal.nth_business_day("2026-01-01", "2026-03-31", 1, cal="XNYS")))
print("last NYSE business day   :",
      show(bcal.nth_business_day("2026-01-01", "2026-03-31", -1, cal="XNYS")))

1st day of the month     : ['2026-01-01', '2026-02-01', '2026-03-01']
last day of the month    : ['2026-01-31', '2026-02-28', '2026-03-31']
1st NYSE business day    : ['2026-01-02', '2026-02-02', '2026-03-02']
last NYSE business day   : ['2026-01-30', '2026-02-27', '2026-03-31']


`month_ends` shows the design point: passing `cal` changes the **question**, not just the
answer. 31 January and 28 February 2026 are Saturdays.

In [18]:
pd.DataFrame(
    {
        "calendar day": show(bcal.month_ends("2026-01-01", "2026-06-30")),
        "business day (NYSE)": show(bcal.month_ends("2026-01-01", "2026-06-30", cal="XNYS")),
        "business day (TARGET2)": show(
            bcal.month_ends("2026-01-01", "2026-06-30", cal="fin:TARGET2")),
    }
)

,calendar day,business day (NYSE),business day (TARGET2)
0,2026-01-31,2026-01-30,2026-01-30
1,2026-02-28,2026-02-27,2026-02-27
2,2026-03-31,2026-03-31,2026-03-31
3,2026-04-30,2026-04-30,2026-04-30
4,2026-05-31,2026-05-29,2026-05-29
5,2026-06-30,2026-06-30,2026-06-30


In [19]:
print("quarters            :", show(bcal.quarter_ends("2026-01-01", "2026-12-31")))
print("fiscal year to Feb  :",
      show(bcal.quarter_ends("2026-01-01", "2026-12-31", anchor_month=2)))
print("years               :", show(bcal.year_ends("2025-01-01", "2027-06-30")))

quarters            : ['2026-03-31', '2026-06-30', '2026-09-30', '2026-12-31']
fiscal year to Feb  : ['2026-02-28', '2026-05-31', '2026-08-31', '2026-11-30']
years               : ['2025-12-31', '2026-12-31']


### IMM dates and option expiries

IMM dates are the 3rd Wednesday of March / June / September / December — that is, of the
quarter's **last month**, not of the quarter. The distinction matters, and it is exactly
what the `months=` filter expresses:

In [20]:
print("3rd Wednesday of the quarter :",
      show(bcal.nth_weekday("2026-01-01", "2026-03-31", 3, WED, freq="Q")))
print("IMM (3rd Wednesday of the month):", show(bcal.imm_dates("2026-01-01", "2026-03-31")))
print()
print("IMM 2026-2027 :", show(bcal.imm_dates("2026-01-01", "2027-12-31")))

3rd Wednesday of the quarter : ['2026-01-21']
IMM (3rd Wednesday of the month): ['2026-03-18']

IMM 2026-2027 : ['2026-03-18', '2026-06-17', '2026-09-16', '2026-12-16', '2027-03-17', '2027-06-16', '2027-09-15', '2027-12-15']


Option expiries are the 3rd Friday, adjusted **backwards** — when the 3rd Friday is Good
Friday, expiry moves back to the Thursday. That was the case in April 2022:

In [21]:
pd.DataFrame(
    {
        "raw 3rd Friday": show(bcal.nth_weekday("2022-01-01", "2022-06-30", 3, FRI)),
        "NYSE expiry": show(bcal.option_expiries("2022-01-01", "2022-06-30", cal="XNYS")),
    }
)

,raw 3rd Friday,NYSE expiry
0,2022-01-21,2022-01-21
1,2022-02-18,2022-02-18
2,2022-03-18,2022-03-18
3,2022-04-15,2022-04-14
4,2022-05-20,2022-05-20
5,2022-06-17,2022-06-17


## 6. `on="edges"` — coupon schedules

Instead of a day **inside** each period, `"edges"` returns the period **boundaries**: one
date more than there are periods. That is what turns `schedule` into a schedule generator.

And the rule that structures the whole module: **the result depends on no calendar at all
until `roll` is passed.**

In [22]:
raw = schedule("2026-02-28", "2027-08-31", "6M", "edges", eom=True)
adjusted = schedule("2026-02-28", "2027-08-31", "6M", "edges", eom=True, cal="XNYS", roll="MF")

pd.DataFrame({"contractual (no roll)": show(raw), "payment (roll=MF)": show(adjusted)})

,contractual (no roll),payment (roll=MF)
0,2026-02-28,2026-02-27
1,2026-08-31,2026-08-31
2,2027-02-28,2027-02-26
3,2027-08-31,2027-08-31


The separation is the reason for the design: a downstream system holding a trade booked
last year needs to know that its 15 March coupon is the same contractual date as yours,
even if a holiday moved when it actually pays. If contractual dates depended on holiday
data, regenerating a snapshot would make the **contract** appear to change.

Here is the proof: with no `roll`, the result is identical across four calendars, even one
with a holiday sitting on a coupon.

In [23]:
from better_calendar import Calendar

calendars = {
    "none (weekday)": None,
    "XNYS": "XNYS",
    "TARGET2": "fin:TARGET2",
    "holiday on the coupon": Calendar("odd", holidays=["2026-08-31", "2026-09-01"]),
}

pd.DataFrame(
    {
        label: {
            "no roll": show(
                schedule("2026-02-28", "2027-08-31", "6M", "edges", eom=True, cal=cal)),
            "roll=MF": show(
                schedule("2026-02-28", "2027-08-31", "6M", "edges", eom=True,
                         cal=cal, roll="MF")),
        }
        for label, cal in calendars.items()
    }
).T

,no roll,roll=MF
none (weekday),"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-31, 2027-02-26, 2027-08-31]"
XNYS,"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-31, 2027-02-26, 2027-08-31]"
TARGET2,"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-31, 2027-02-26, 2027-08-31]"
holiday on the coupon,"[2026-02-28, 2026-08-31, 2027-02-28, 2027-08-31]","[2026-02-27, 2026-08-28, 2027-02-26, 2027-08-31]"


### Stubs

A schedule rarely divides evenly. The *stub* is what you do with the remainder. A five-month
term at a quarterly frequency leaves two months to place:

In [24]:
pd.DataFrame(
    [
        {"stub": stub,
         "dates": show(schedule("2026-01-15", "2026-06-15", "3M", "edges", stub=stub))}
        for stub in ("short_front", "long_front", "short_back", "long_back")
    ]
).set_index("stub")

,dates
stub,
short_front,"[2026-01-15, 2026-03-15, 2026-06-15]"
long_front,"[2026-01-15, 2026-06-15]"
short_back,"[2026-01-15, 2026-04-15, 2026-06-15]"
long_back,"[2026-01-15, 2026-06-15]"


The stub choice determines **which end** the regular grid is measured from: a *front* stub
anchors on the end date and generates backwards, a *back* stub on the start and forwards.
That is what makes coupons land on maturity rather than drifting away from it.

In [25]:
print("front (anchored on the end)  :",
      show(schedule("2026-01-10", "2027-01-15", "6M", "edges", stub="short_front")))
print("back  (anchored on the start):",
      show(schedule("2026-01-10", "2027-01-15", "6M", "edges", stub="short_back")))

front (anchored on the end)  : ['2026-01-10', '2026-01-15', '2026-07-15', '2027-01-15']
back  (anchored on the start): ['2026-01-10', '2026-07-10', '2027-01-10', '2027-01-15']


In [26]:
# `none` refuses a term that does not divide evenly rather than inventing a stub.
try:
    schedule("2026-01-15", "2026-06-15", "3M", "edges", stub="none")
except bcal.ScheduleError as exc:
    print(exc)

2026-01-15 to 2026-06-15 is not a whole number of 3M periods, and stub='none' forbids a stub. Choose a stub convention, or move one of the dates.


### Dates are measured from the anchor, never step by step

Otherwise 31 January would slip to 28 February and then **stay** on the 28th for the life of
the trade. Here it returns to the 31st as soon as the month allows:

In [27]:
show(schedule("2026-01-31", "2026-06-30", "1M", "edges", stub="short_back"))

['2026-01-31',
 '2026-02-28',
 '2026-03-31',
 '2026-04-30',
 '2026-05-31',
 '2026-06-30']

### Accrual periods

`periods()` returns the intervals between consecutive dates, half-open `[start, end)` so
that periods tile without counting the boundary day twice.

In [28]:
pd.DataFrame(
    [
        {"start": p.start, "end": p.end, "calendar days": len(p),
         "business days": len(p.business_days("XNYS"))}
        for p in periods("2026-01-15", "2027-01-15", "3M", cal="XNYS", roll="MF")
    ]
)

,start,end,calendar days,business days
0,2026-01-15,2026-04-15,90,61
1,2026-04-15,2026-07-15,91,62
2,2026-07-15,2026-10-15,92,65
3,2026-10-15,2027-01-15,92,63


`periods()` takes any selector, not just `"edges"`:

In [29]:
monthly = periods("2026-01-01", "2026-06-30", "M", "last")
print(f"{len(monthly)} intervals, month end to month end")
print([(str(p.start), str(p.end)) for p in monthly[:3]])

5 intervals, month end to month end
[('2026-01-31', '2026-02-28'), ('2026-02-28', '2026-03-31'), ('2026-03-31', '2026-04-30')]


## 7. A complete case: bond coupons

Semi-annual, from 15 March 2026 to 15 March 2031, settling in the euro area.

In [30]:
contractual = schedule("2026-03-15", "2031-03-15", "6M", "edges")
payment = schedule("2026-03-15", "2031-03-15", "6M", "edges",
                   cal="fin:TARGET2", roll="MF")

coupons = pd.DataFrame({"contractual": show(contractual), "payment": show(payment)})
coupons["moved?"] = coupons["contractual"] != coupons["payment"]
coupons["business days since previous"] = [None] + [
    bcal.count(a, b, cal="fin:TARGET2")
    for a, b in zip(coupons["payment"], coupons["payment"][1:])
]
coupons

,contractual,payment,moved?,business days since previous
0,2026-03-15,2026-03-16,True,NaN
1,2026-09-15,2026-09-15,False,128.0
2,2027-03-15,2027-03-15,False,127.0
3,2027-09-15,2027-09-15,False,130.0
4,2028-03-15,2028-03-15,False,130.0
5,2028-09-15,2028-09-15,False,129.0
6,2029-03-15,2029-03-15,False,126.0
7,2029-09-15,2029-09-17,True,129.0
8,2030-03-15,2030-03-15,False,126.0
9,2030-09-15,2030-09-16,True,128.0


## Recap

| Call | Role |
|---|---|
| `schedule(a, b, every, on, …)` | the engine: cut, then choose |
| `every` | `"D"` `"W"` `"M"` `"Q"` `"Y"` or a multiple (`"3M"`) |
| `on` | `"last"`, `"15"`, `"1 B"`, `"2 THU"`, `"edges"`, or a list |
| `missing=` | `"skip"` (default), `"clamp"`, `"raise"` |
| `months=` | restrict to certain calendar months (IMM) |
| `stub=`, `eom=` | only with `on="edges"` |
| `periods(...)` | the intervals between consecutive dates |
| `nth_weekday`, `last_weekday`, `nth_day`, `nth_business_day` | named shortcuts |
| `month_ends`, `quarter_ends`, `year_ends`, `imm_dates`, `option_expiries` | likewise |
| `MON`…`SUN`, `Weekday`, `Nth` | constants and the typed selector form |

**Next:** [05 — Timezones and sessions](05-timezones-and-sessions.ipynb)